<a href="https://colab.research.google.com/github/pundravadasnigdha/disaster-trend-intelligence-chatbot/blob/main/Copy_of_disaster_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q streamlit pandas openpyxl ollama

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 98 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (8,134 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(["ollama", "serve"])

time.sleep(5)

print("Ollama started")

Ollama started


In [ ]:
!ollama pull qwen2.5:0.5b

In [ ]:
!ollama list

NAME            ID              SIZE      MODIFIED               
qwen2.5:0.5b    a8b0c5157701    397 MB    Less than a second ago    


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving emdat_dataset_1978to2025.xlsx to emdat_dataset_1978to2025.xlsx


In [ ]:
%%writefile llm.py
import ollama
import json
import re


def extract_query(user_query):
    q = user_query.lower()

    prompt = f"""
You are an information extraction system for a disaster analytics chatbot.

Return ONLY valid JSON. No explanation.

Extract:
1. disaster_type as a list
2. location as a list
3. time as an object
4. intent
5. is_followup
6. query_plan

Allowed disaster_type values:
- drought
- earthquake
- epidemic
- extreme temperature
- flood
- glacial lake outburst flood
- infestation
- mass movement (wet)
- mass movement (dry)
- storm
- wildfire
- lightning

Disaster rules:
- cyclone/cyclones = storm
- forest fire/forest fires = wildfire
- landslide/landslides = mass movement (wet)
- heatwave/cold wave = extreme temperature
- locust/pest = infestation
- no disaster mentioned = []

Location rules:
- Extract Indian states, cities, and India if mentioned
- no location = []

Time schema:
"time": {{
    "type": "relative | absolute | year | month | range | none",
    "value": "string or null"
}}

Time rules:
- today/recent/last 24 hours/this week/this month/this year/last month/last year/yesterday = relative
- 2020 = year
- May 2026 = month
- 28 October 2020 = absolute
- between 2020 and 2022 = range
- no time = none

Intent rules:
- records/show/find/events/occurred = records
- trend/trends = trend
- frequency/count/how many = frequency
- affected/people/population = affected
- deaths/died = deaths
- magnitude = magnitude
- damage/loss/money/cost = damage
- table/display/all records = table
- details/summary/more information/tell me more = summary
- otherwise = records

Follow-up:
If the user asks about previously selected records, set is_followup true.
Examples:
- how many people were affected
- what is the magnitude
- what was the damage
- where did it happen
- which month
- show all records in table
- give all available information

Query plan:
"query_plan": {{
    "operation": "records | frequency | trend | sum | list | table | summary",
    "columns_needed": [],
    "aggregation": "none | sum | count | list"
}}

User Query:
{user_query}

JSON format:
{{
    "disaster_type": [],
    "location": [],
    "time": {{
        "type": "none",
        "value": null
    }},
    "intent": "records",
    "is_followup": false,
    "query_plan": {{
        "operation": "records",
        "columns_needed": [],
        "aggregation": "none"
    }}
}}
"""

    try:
        response = ollama.chat(
            model="tinyllama:latest",
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0, "num_predict": 180}
        )

        content = response["message"]["content"]
        start = content.find("{")
        end = content.rfind("}") + 1

        if start != -1 and end != 0:
            data = json.loads(content[start:end])
        else:
            data = {}

    except:
        data = {}

    # ---------------- SAFE FALLBACK ----------------
    disaster_patterns = [
        (r"\bglacial lake outburst flood\b|\bglof\b", "glacial lake outburst flood"),
        (r"\bfloods?\b", "flood"),
        (r"\bearthquakes?\b|\btremors?\b", "earthquake"),
        (r"\bcyclones?\b|\bstorms?\b|\bhurricanes?\b", "storm"),
        (r"\blandslides?\b|\bmudslides?\b", "mass movement (wet)"),
        (r"\bforest fires?\b|\bwildfires?\b", "wildfire"),
        (r"\blightning\b", "lightning"),
        (r"\bepidemics?\b", "epidemic"),
        (r"\bheatwaves?\b|\bcold waves?\b", "extreme temperature"),
        (r"\bdroughts?\b", "drought"),
        (r"\blocusts?\b|\bpests?\b|\binfestation\b", "infestation"),
    ]

    disasters = []
    for pattern, value in disaster_patterns:
        for match in re.finditer(pattern, q):
            disasters.append((match.start(), value))

    disasters.sort(key=lambda x: x[0])

    final_disasters = []
    seen = set()

    for _, disaster in disasters:
        if disaster not in seen:
            final_disasters.append(disaster)
            seen.add(disaster)

    data["disaster_type"] = final_disasters

    known_locations = [
        "andhra pradesh", "telangana", "tamil nadu", "kerala",
        "odisha", "gujarat", "west bengal", "assam", "bihar",
        "maharashtra", "karnataka", "rajasthan", "uttar pradesh",
        "madhya pradesh", "himachal pradesh", "delhi", "chennai",
        "hyderabad", "mumbai", "india"
    ]

    locations = []
    for loc in known_locations:
        if loc in q:
            locations.append(loc.title())

    data["location"] = locations

    data["time"] = {
        "type": "none",
        "value": None
    }

    if "last 24 hours" in q:
        data["time"] = {"type": "relative", "value": "last 24 hours"}
    elif "today" in q:
        data["time"] = {"type": "relative", "value": "today"}
    elif "recent" in q:
        data["time"] = {"type": "relative", "value": "recent"}
    elif "this week" in q:
        data["time"] = {"type": "relative", "value": "this week"}
    elif "this month" in q:
        data["time"] = {"type": "relative", "value": "this month"}
    elif "this year" in q:
        data["time"] = {"type": "relative", "value": "this year"}
    elif "last month" in q:
        data["time"] = {"type": "relative", "value": "last month"}
    elif "last year" in q:
        data["time"] = {"type": "relative", "value": "last year"}
    elif "yesterday" in q:
        data["time"] = {"type": "relative","value": "yesterday"}

    between_match = re.search(r"between\s+(\d{4})\s+and\s+(\d{4})", q)

    if between_match:
        data["time"] = {
            "type": "range",
            "value": f"{between_match.group(1)}-{between_match.group(2)}"
        }

    date_match = re.search(
        r"(\d{1,2})\s*"
        r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|"
        r"january|february|march|april|june|july|august|"
        r"september|october|november|december)"
        r"\s*(\d{4})",
        q
    )

    if date_match:
        data["time"] = {
            "type": "absolute",
            "value": f"{date_match.group(1)} {date_match.group(2)} {date_match.group(3)}"
        }

    month_year_match = re.search(
        r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|"
        r"january|february|march|april|june|july|august|"
        r"september|october|november|december)"
        r"\s*,?\s*(\d{4})",
        q
    )

    if month_year_match and data["time"]["type"] != "absolute":
        data["time"] = {
            "type": "month",
            "value": f"{month_year_match.group(1)} {month_year_match.group(2)}"
        }

    year_match = re.search(r"\b(19|20)\d{2}\b", q)

    if year_match and data["time"]["type"] == "none":
        data["time"] = {
            "type": "year",
            "value": year_match.group()
        }

    if "trend" in q:
        intent = "trend"
    elif "frequency" in q or "count" in q or "how many" in q:
        intent = "frequency"
    elif "affected" in q or "people" in q or "population" in q:
        intent = "affected"
    elif "death" in q or "deaths" in q or "died" in q:
        intent = "deaths"
    elif "magnitude" in q:
        intent = "magnitude"
    elif "damage" in q or "money" in q or "loss" in q or "cost" in q:
        intent = "damage"
    elif "table" in q or "display" in q or "all records" in q:
        intent = "table"
    elif "summary" in q or "details" in q or "tell me more" in q or "more information" in q:
        intent = "summary"
    else:
        intent = data.get("intent", "records")

    data["intent"] = intent

    if intent in ["affected", "deaths", "damage"]:
        operation = "sum"
    elif intent in ["magnitude"]:
        operation = "list"
    elif intent == "table":
        operation = "table"
    elif intent == "summary":
        operation = "summary"
    elif intent == "trend":
        operation = "trend"
    elif intent == "frequency":
        operation = "frequency"
    else:
        operation = "records"

    data["is_followup"] = (
        not final_disasters
        and not locations
        and data["time"]["type"] == "none"
        and intent != "records"
    )

    data["query_plan"] = {
        "operation": operation,
        "columns_needed": [],
        "aggregation": "none"
    }

    return data

Writing llm.py


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import tempfile
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from llm import extract_query

st.set_page_config(page_title="Disaster Trend Intelligence Chatbot", layout="wide")

st.markdown("""
<style>
.user-msg {
    background-color: #e8f0fe;
    padding: 14px 18px;
    border-radius: 18px;
    margin: 12px 0;
    max-width: 70%;
    text-align: left;
}
.bot-msg {
    background-color: #dcf8c6;
    padding: 14px 18px;
    border-radius: 18px;
    margin: 12px 0 12px auto;
    max-width: 75%;
    text-align: left;
}
.table-container {
    overflow-x: auto;
    max-height: 450px;
    overflow-y: auto;
}
table {
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
    background-color: white;
}
th, td {
    border: 1px solid #ccc;
    padding: 8px;
    text-align: left;
}
th {
    background-color: #f2f2f2;
}
</style>
""", unsafe_allow_html=True)

st.title("🌍 Disaster Trend Intelligence Chatbot")
st.write("Ask disaster-related questions using text or voice.")

df = pd.read_excel("/content/drive/MyDrive/emdat_dataset_1978to2025.xlsx")
df = df.fillna("")

if "messages" not in st.session_state:
    st.session_state.messages = []

if "context_result" not in st.session_state:
    st.session_state.context_result = None

if "voice_text" not in st.session_state:
    st.session_state.voice_text = None

if "audio_bytes" not in st.session_state:
    st.session_state.audio_bytes = None

if "consumed_audio_bytes" not in st.session_state:
    st.session_state.consumed_audio_bytes = None


@st.cache_resource
def load_asr_pipeline():
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model_id = "openai/whisper-small"

    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True
    )

    model.to(device)

    processor = AutoProcessor.from_pretrained(model_id)

    return pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=torch_dtype,
        device=device
    )


def transcribe_audio(audio_file):
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as temp_audio:
        temp_audio.write(audio_file.getvalue())
        audio_path = temp_audio.name

    pipe = load_asr_pipeline()
    result = pipe(audio_path)

    return result["text"]


def clean(value):
    if value == "" or pd.isna(value):
        return "Not available"

    if isinstance(value, float) and value.is_integer():
        return int(value)

    return value


def month_name(value):
    months = {
        1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
        5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
        9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
    }

    try:
        return months.get(int(float(value)), "Not available")
    except:
        return "Not available"


def month_number(month_text):
    months = {
        "jan": 1, "january": 1,
        "feb": 2, "february": 2,
        "mar": 3, "march": 3,
        "apr": 4, "april": 4,
        "may": 5,
        "jun": 6, "june": 6,
        "jul": 7, "july": 7,
        "aug": 8, "august": 8,
        "sep": 9, "september": 9,
        "oct": 10, "october": 10,
        "nov": 11, "november": 11,
        "dec": 12, "december": 12
    }

    return months.get(str(month_text).lower())


def get_location_text(row):
    loc = str(row.get("Location", "")).strip()
    return loc if loc else "Not available"


def magnitude_text(row):
    mag = row.get("Magnitude", "")
    scale = row.get("Magnitude Scale", "")

    if mag == "" or pd.isna(mag):
        return "Not available"

    if scale == "" or pd.isna(scale):
        return str(clean(mag))

    return f"{clean(mag)} {scale}"


def filter_database(extracted):
    disasters = extracted.get("disaster_type", [])
    locations = extracted.get("location", [])
    time_info = extracted.get("time", {"type": "none", "value": None})

    result = df.copy()

    if disasters:
        disaster_matches = []

        for disaster in disasters:
            mask = (
                result["Disaster Type"]
                .astype(str)
                .str.lower()
                .str.contains(disaster.lower(), na=False, regex=False)
            )

            if "Disaster Subtype" in result.columns:
                mask = mask | (
                    result["Disaster Subtype"]
                    .astype(str)
                    .str.lower()
                    .str.contains(disaster.lower(), na=False, regex=False)
                )

            disaster_matches.append(result[mask])

        result = pd.concat(disaster_matches).drop_duplicates()

    if time_info and time_info.get("type") != "none":
        time_type = time_info.get("type")
        time_value = time_info.get("value")
        year_col = pd.to_numeric(result["Start Year"], errors="coerce")

        if time_type == "relative":
            return pd.DataFrame(columns=result.columns)

        if time_type == "year":
            result = result[year_col == int(time_value)]

        elif time_type == "range":
            start_year, end_year = time_value.split("-")
            result = result[
                (year_col >= int(start_year)) &
                (year_col <= int(end_year))
            ]

        elif time_type == "month":
            parts = time_value.split()
            month = month_number(parts[0])
            year = int(parts[1])
            month_col = pd.to_numeric(result["Start Month"], errors="coerce")

            result = result[
                (year_col == year) &
                (month_col == month)
            ]

        elif time_type == "absolute":
            parts = time_value.split()
            day = int(parts[0])
            month = month_number(parts[1])
            year = int(parts[2])

            month_col = pd.to_numeric(result["Start Month"], errors="coerce")
            day_col = pd.to_numeric(result["Start Day"], errors="coerce")

            result = result[
                (year_col == year) &
                (month_col == month) &
                (day_col == day)
            ]

    if locations:
        filtered_locations = [
            loc for loc in locations
            if loc.lower() != "india"
        ]

        if filtered_locations:
            search_columns = [
                col for col in ["Location", "Admin Units", "GADM Admin Units"]
                if col in result.columns
            ]

            matches = []

            for loc in filtered_locations:
                loc_mask = pd.Series(False, index=result.index)

                for col in search_columns:
                    loc_mask = loc_mask | (
                        result[col]
                        .astype(str)
                        .str.contains(loc, case=False, na=False, regex=False)
                    )

                matches.append(result[loc_mask])

            result = pd.concat(matches).drop_duplicates()

    return result


def table_response(result):
    display_cols = [
        "Disaster Type",
        "Disaster Subtype",
        "Location",
        "Start Year",
        "Start Month",
        "Start Day",
        "Total Deaths",
        "Total Affected",
        "Magnitude",
        "Magnitude Scale",
        "Total Damage ('000 US$)"
    ]

    display_cols = [col for col in display_cols if col in result.columns]
    table_df = result[display_cols].copy()

    if "Start Month" in table_df.columns:
        table_df["Start Month"] = table_df["Start Month"].apply(month_name)

    for col in table_df.columns:
        table_df[col] = table_df[col].apply(clean)

    return f"<div class='table-container'>{table_df.to_html(index=False, escape=False)}</div>"


def single_record_summary(result):
    row = result.iloc[0]

    return f"""
I found 1 matching record from the database.<br><br>
<b>Disaster Summary</b><br><br>
• Disaster Type: {clean(row.get("Disaster Type", ""))}<br>
• Disaster Subtype: {clean(row.get("Disaster Subtype", ""))}<br>
• Location: {get_location_text(row)}<br>
• Start Time: {month_name(row.get("Start Month", ""))} {clean(row.get("Start Year", ""))}<br>
• Start Day: {clean(row.get("Start Day", ""))}<br>
• Total Deaths: {clean(row.get("Total Deaths", ""))}<br>
• Total Affected: {clean(row.get("Total Affected", ""))}<br>
• Magnitude: {magnitude_text(row)}<br>
• Total Damage: {clean(row.get("Total Damage ('000 US$)", ""))} ('000 US$)
"""


def count_response(result, extracted):
    disasters = extracted.get("disaster_type", [])
    locations = extracted.get("location", [])
    time_info = extracted.get("time", {"type": "none", "value": None})

    disaster_text = ", ".join(disasters) if disasters else "disaster"
    location_text = ", ".join(locations) if locations else "India"

    time_text = ""
    if time_info and time_info.get("type") != "none":
        time_text = f" for {time_info.get('value')}"

    return (
        f"I found {len(result)} {disaster_text} record(s) "
        f"in {location_text}{time_text}. "
        f"You can ask any follow-up question about these records."
    )


def find_requested_columns(query, result):
    q = query.lower()
    requested = []

    aliases = {
        "Total Affected": ["affected", "people", "population"],
        "Total Deaths": ["death", "deaths", "died"],
        "Magnitude": ["magnitude"],
        "Magnitude Scale": ["scale"],
        "Total Damage ('000 US$)": ["damage", "money", "loss", "cost"],
        "Location": ["location", "area", "areas", "where"],
        "Disaster Type": ["disaster type", "type of disaster", "disaster", "disasters"],
        "Disaster Subtype": ["subtype", "kind", "type of flood", "type of cyclone"],
        "Start Year": ["year", "years"],
        "Start Month": ["month", "months"],
        "Start Day": ["day", "date"]
    }

    for col, words in aliases.items():
        if col in result.columns:
            for word in words:
                if word in q:
                    requested.append(col)
                    break

    for col in result.columns:
        col_words = col.lower().replace("(", "").replace(")", "").replace("'", "").split()
        if all(word in q for word in col_words[:2]):
            requested.append(col)

    return list(dict.fromkeys(requested))


def answer_from_context(query):
    result = st.session_state.context_result

    if result is None or len(result) == 0:
        return None

    q = query.lower()

    if "table" in q or "display" in q or "show all" in q or "all records" in q:
        return table_response(result)

    if "summary" in q or "details" in q or "tell me more" in q or "all available information" in q:
        if len(result) == 1:
            return single_record_summary(result)
        return table_response(result.head(20))

    requested_cols = find_requested_columns(query, result)

    if not requested_cols:
        return "I can answer from the selected records. Ask about any column like year, month, location, deaths, affected, damage, magnitude, subtype, or ask for table."

    numeric_cols = []
    list_cols = []

    for col in requested_cols:
        numeric_values = pd.to_numeric(result[col], errors="coerce")
        if not numeric_values.dropna().empty and col in [
            "Total Affected",
            "Total Deaths",
            "Total Damage ('000 US$)"
        ]:
            numeric_cols.append(col)
        else:
            list_cols.append(col)

    lines = ["Here is the information from the selected records:"]

    for col in numeric_cols:
        values = pd.to_numeric(result[col], errors="coerce")
        lines.append(f"• {col}: {int(values.fillna(0).sum())}")

    for col in list_cols:
        values = (
            result[col]
            .replace("", pd.NA)
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        if col == "Start Month":
            values = [month_name(v) for v in values]

        if values:
            lines.append(f"<br><b>{col}</b>:")
            lines.extend([f"• {v}" for v in values[:30]])
        else:
            lines.append(f"<br><b>{col}</b>: Not available")

    return "<br>".join(lines)


def is_probably_followup(query, extracted):
    if st.session_state.context_result is None:
        return False

    if extracted.get("disaster_type") or extracted.get("location"):
        return False

    time_info = extracted.get("time", {"type": "none", "value": None})
    if time_info and time_info.get("type") != "none":
        return False

    q = query.lower()

    followup_markers = [
        "it", "this", "that", "these", "those", "same",
        "affected", "people", "population",
        "death", "deaths", "died",
        "magnitude", "damage", "money", "loss", "cost",
        "area", "areas", "where", "location",
        "year", "years", "month", "months", "day", "date",
        "subtype", "type", "kind",
        "table", "display", "show all", "all records",
        "summary", "details", "tell me more", "all available information"
    ]

    return any(marker in q for marker in followup_markers)


def make_response(query):
    extracted = extract_query(query)

    if is_probably_followup(query, extracted):
        return answer_from_context(query)

    time_info = extracted.get("time", {"type": "none", "value": None})

    if time_info and time_info.get("type") == "relative":
        st.session_state.context_result = None
        return "No recent records available in the historical dataset."

    result = filter_database(extracted)

    if len(result) == 0:
        return "No matching records were found for your query."

    st.session_state.context_result = result

    intent = extracted.get("intent", "records")

    if intent == "trend":
        yearly_counts = result.groupby("Start Year").size().sort_index().to_dict()
        trend_text = "<br>".join(
            [f"• {int(year)}: {freq} record(s)" for year, freq in yearly_counts.items()]
        )
        return f"I found {len(result)} matching record(s).<br><br>Year-wise trend:<br>{trend_text}"

    if intent == "frequency":
        return count_response(result, extracted)

    if intent in ["affected", "deaths", "magnitude", "damage", "table", "summary"]:
        return answer_from_context(query)

    if len(result) == 1:
        return single_record_summary(result)

    return count_response(result, extracted)


for message in st.session_state.messages:
    css_class = "user-msg" if message["role"] == "user" else "bot-msg"

    st.markdown(
        f"""
        <div class="{css_class}">
            {message["content"]}
        </div>
        """,
        unsafe_allow_html=True
    )


st.markdown("### Ask your query")

col1, col2 = st.columns([8, 1])

with col1:
    typed_input = st.chat_input("Ask about disaster records...")

with col2:
    voice_audio = st.audio_input("🎙️", label_visibility="collapsed")

use_voice = False
discard_voice = False

if voice_audio is not None:
    current_audio_bytes = voice_audio.getvalue()

    if current_audio_bytes != st.session_state.consumed_audio_bytes:
        if st.session_state.audio_bytes != current_audio_bytes:
            st.session_state.audio_bytes = current_audio_bytes
            st.session_state.voice_text = None

        if st.session_state.voice_text is None:
            with st.spinner("Transcribing voice query..."):
                st.session_state.voice_text = transcribe_audio(voice_audio)

if st.session_state.voice_text:
    edited_voice_text = st.text_input(
        "Transcribed voice query",
        value=st.session_state.voice_text
    )

    st.session_state.voice_text = edited_voice_text

    col_use, col_discard = st.columns(2)

    with col_use:
        use_voice = st.button("✅ Use Query")

    with col_discard:
        discard_voice = st.button("❌ Discard")

if discard_voice:
    st.session_state.voice_text = None
    st.session_state.audio_bytes = None

    if voice_audio is not None:
        st.session_state.consumed_audio_bytes = voice_audio.getvalue()

    st.rerun()

user_input = None

if use_voice and st.session_state.voice_text:
    user_input = st.session_state.voice_text

elif typed_input:
    user_input = typed_input

if user_input:
    st.session_state.messages.append(
        {"role": "user", "content": user_input}
    )

    response = make_response(user_input)

    response = response.replace("**", "")
    response = response.replace("__", "")
    response = response.strip()

    st.session_state.messages.append(
        {"role": "assistant", "content": response}
    )

    if use_voice:
        st.session_state.consumed_audio_bytes = st.session_state.audio_bytes
        st.session_state.voice_text = None
        st.session_state.audio_bytes = None

    st.rerun()

Writing app.py


In [ ]:
!pkill streamlit

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!cat /content/logs.txt



2026-06-23 17:47:30.164 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.228.23.210:8501



In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
changed 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

In [ ]:
get_ipython().system_raw('lt --port 8501 > tunnel.txt 2>&1 &')

In [ ]:
!cat tunnel.txt

your url is: https://brave-loops-brush.loca.lt


In [ ]:
!ls

app.py	emdat_dataset_1978to2025.xlsx  logs.txt     tunnel.txt
drive	llm.py			       sample_data


In [ ]:
!pip install -q streamlit pandas openpyxl ollama transformers accelerate librosa soundfile